In [1]:
DEPTH = "../../../zed2i_depth_image/"
RGB = "../../../zed2i_right_images/"

In [2]:
import numpy as np
from PIL import Image
d = np.array(Image.open(DEPTH+"000000.png"))
print(d.dtype, d.min(), d.max())

uint16 0 15000


TF Tree

In [3]:
import json
with open("../../../tf/.zattrs") as f:
    attrs = json.load(f)
print(json.dumps(attrs, indent=2))

{
  "tf": {
    "adis16475_imu": {
      "base_frame_id": "box_base",
      "child_frame_id": "adis16475_imu",
      "rotation": {
        "w": 0.005251232714324072,
        "x": 0.7046754024829419,
        "y": -0.7094799413019802,
        "z": -0.006573779782016809
      },
      "translation": {
        "x": -0.015846751421098565,
        "y": 0.19322607669094574,
        "z": 0.07956850717224997
      }
    },
    "alphasense_base": {
      "base_frame_id": "box_base",
      "child_frame_id": "alphasense_base",
      "rotation": {
        "w": 0.541225186106887,
        "x": -0.45494336062045815,
        "y": -0.4544944669145965,
        "z": -0.5417901956430607
      },
      "translation": {
        "x": 0.04600713143181777,
        "y": -0.0032140911178705073,
        "z": -0.3751986320280252
      }
    },
    "alphasense_front_center": {
      "base_frame_id": "box_base",
      "child_frame_id": "alphasense_front_center",
      "rotation": {
        "w": 0.541225186106887,
   

Static Pointcloud

In [4]:
import open3d as o3d

rgb = np.array(Image.open(RGB + "000000.jpeg").convert("RGB"))
depth = np.array(Image.open(DEPTH + "000000.png"))  # uint16, mm

h, w = depth.shape
fx, fy, cx, cy = 1052.19970703125, 1052.19970703125, 956.3457641601562, 553.95703125

rgb_o3d = o3d.geometry.Image(rgb)
depth_o3d = o3d.geometry.Image(depth)

rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
    rgb_o3d, depth_o3d,
    depth_scale=1000.0,   # mm -> meters
    depth_trunc=30.0,     # clip far range, adjust to your terrain scenario
    convert_rgb_to_intensity=False
)

intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, cx, cy)
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)

# ZED/OpenCV convention has Y-down, Z-forward; Open3D expects Y-up, Z-backward for standard viewing
pcd.transform([[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]])

o3d.visualization.draw_geometries([pcd])

Dynamic Pointcloud

In [ ]:
import open3d as o3d
import numpy as np
from PIL import Image
import glob
import os
import time
import re

# --- Config ---
fx, fy, cx, cy = 1052.19970703125, 1052.19970703125, 956.3457641601562, 553.95703125
depth_scale = 1000.0   # mm -> meters
depth_trunc = 30.0
frame_delay = 0.05     # seconds between frames (set to 0 for as-fast-as-possible)

# ZED/OpenCV -> Open3D display convention flip
FLIP = np.array([[1, 0, 0, 0],
                  [0, -1, 0, 0],
                  [0, 0, -1, 0],
                  [0, 0, 0, 1]])

# --- Gather file list (sorted numerically by frame id) ---
def frame_id(path):
    m = re.search(r'(\d+)', os.path.basename(path))
    return int(m.group(1)) if m else 0

rgb_files = sorted(glob.glob(os.path.join(RGB, "*.jpeg")), key=frame_id)
depth_files = sorted(glob.glob(os.path.join(DEPTH, "*.png")), key=frame_id)

assert len(rgb_files) == len(depth_files), "Mismatch between RGB and depth frame counts"
print(f"Found {len(rgb_files)} frame pairs")

def build_pcd(rgb_path, depth_path):
    rgb = np.array(Image.open(rgb_path).convert("RGB"))
    depth = np.array(Image.open(depth_path))  # uint16, mm

    h, w = depth.shape
    rgb_o3d = o3d.geometry.Image(rgb)
    depth_o3d = o3d.geometry.Image(depth)

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb_o3d, depth_o3d,
        depth_scale=depth_scale,
        depth_trunc=depth_trunc,
        convert_rgb_to_intensity=False
    )

    intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, cx, cy)
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    pcd.transform(FLIP)
    return pcd

# --- Set up visualizer ---
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="ZED2i Point Cloud Sequence", width=1280, height=720)

# Initialize with first frame
pcd = build_pcd(rgb_files[0], depth_files[0])
vis.add_geometry(pcd)

# Optional: lock the camera view so it doesn't reset each frame
view_ctl = vis.get_view_control()

for i in range(1, len(rgb_files)):
    new_pcd = build_pcd(rgb_files[i], depth_files[i])

    pcd.points = new_pcd.points
    pcd.colors = new_pcd.colors

    vis.update_geometry(pcd)
    if not vis.poll_events():
        break  # window was closed
    vis.update_renderer()

    time.sleep(frame_delay)
    print(f"\rFrame {i+1}/{len(rgb_files)}", end="")

print("\nDone.")
vis.destroy_window()

Found 4949 frame pairs
Frame 1542/4949
Done.


: 

Static Transformed Pointcloud

In [8]:
import numpy as np
import open3d as o3d
from PIL import Image
from scipy.spatial.transform import Rotation as R

# 1. Load Images & Setup Intrinsic Matrix
rgb = np.array(Image.open(RGB + "000100.jpeg").convert("RGB"))
depth = np.array(Image.open(DEPTH + "000100.png"))  # uint16, mm

h, w = depth.shape
fx, fy, cx, cy = 1052.19970703125, 1052.19970703125, 956.3457641601562, 553.95703125

rgb_o3d = o3d.geometry.Image(rgb)
depth_o3d = o3d.geometry.Image(depth)

rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
    rgb_o3d, depth_o3d,
    depth_scale=1000.0,   # mm -> meters
    depth_trunc=30.0,     # clip far range
    convert_rgb_to_intensity=False
)

intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, cx, cy)
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)

# 2. Build Transformation Matrix from JSON (zed2i_right to box_base)
# Note: SciPy Rotation takes quaternion as [x, y, z, w]
quat = [-0.45720033826192996, -0.4523183402260819, -0.5455934690231464, 0.5373115821826182]
trans = [-0.012769940823864567, -0.07826319215076401, -0.39482353374563656]

T_boxbase_zed2i = np.eye(4)
T_boxbase_zed2i[:3, :3] = R.from_quat(quat).as_matrix()
T_boxbase_zed2i[:3, 3] = trans

# 3. Handle Optical-to-Body Frame Alignment (Standard ROS convention)
# Open3D output is in Camera Optical Frame (+X Right, +Y Down, +Z Forward).
# Standard ROS camera body frame is (+X Forward, +Y Left, +Z Up).
T_optical_to_body = np.array([
    [ 0,  0,  1,  0],
    [-1,  0,  0,  0],
    [ 0, -1,  0,  0],
    [ 0,  0,  0,  1]
])

# Combine transforms: Optical -> ZED Body -> Box Base
T_total = T_boxbase_zed2i @ T_optical_to_body

# Apply total transformation to point cloud
pcd.transform(T_total)

# 4. Visualize Point Cloud in box_base Frame
# Adds coordinate axes (Red = X, Green = Y, Blue = Z) at box_base origin (0,0,0)
axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5, origin=[0, 0, 0])
o3d.visualization.draw_geometries([pcd, axes])

Dynamic Transformed Pointcloud

In [ ]:
import open3d as o3d
import numpy as np
from PIL import Image
from scipy.spatial.transform import Rotation as R
import glob
import os
import time
import re


fx, fy, cx, cy = 1052.19970703125, 1052.19970703125, 956.3457641601562, 553.95703125
depth_scale = 1000.0   # mm -> meters
depth_trunc = 30.0
frame_delay = 0.05     # seconds between frames

# --- Compute Transformation Matrix to box_base ---
# Quaternion in SciPy format: [x, y, z, w]
quat = [-0.45720033826192996, -0.4523183402260819, -0.5455934690231464, 0.5373115821826182]
trans = [-0.012769940823864567, -0.07826319215076401, -0.39482353374563656]

T_boxbase_zed2i = np.eye(4)
T_boxbase_zed2i[:3, :3] = R.from_quat(quat).as_matrix()
T_boxbase_zed2i[:3, 3] = trans

# ROS Optical (+X right, +Y down, +Z forward) to Body frame alignment
T_optical_to_body = np.array([
    [ 0,  0,  1,  0],
    [-1,  0,  0,  0],
    [ 0, -1,  0,  0],
    [ 0,  0,  0,  1]
])

# Combine into single static transform
T_TOTAL = T_boxbase_zed2i @ T_optical_to_body

# --- Gather file list (sorted numerically by frame id) ---
def frame_id(path):
    m = re.search(r'(\d+)', os.path.basename(path))
    return int(m.group(1)) if m else 0

rgb_files = sorted(glob.glob(os.path.join(RGB, "*.jpeg")), key=frame_id)
depth_files = sorted(glob.glob(os.path.join(DEPTH, "*.png")), key=frame_id)

assert len(rgb_files) == len(depth_files), "Mismatch between RGB and depth frame counts"
print(f"Found {len(rgb_files)} frame pairs")

def build_pcd(rgb_path, depth_path):
    rgb = np.array(Image.open(rgb_path).convert("RGB"))
    depth = np.array(Image.open(depth_path))  # uint16, mm

    h, w = depth.shape
    rgb_o3d = o3d.geometry.Image(rgb)
    depth_o3d = o3d.geometry.Image(depth)

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb_o3d, depth_o3d,
        depth_scale=depth_scale,
        depth_trunc=depth_trunc,
        convert_rgb_to_intensity=False
    )

    intrinsic = o3d.camera.PinholeCameraIntrinsic(w, h, fx, fy, cx, cy)
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)
    
    # Transform from camera optical frame to box_base frame
    pcd.transform(T_TOTAL)
    return pcd

# --- Set up visualizer ---
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="ZED2i Point Cloud Sequence (box_base Frame)", width=1280, height=720)

# Initialize with first frame
pcd = build_pcd(rgb_files[0], depth_files[0])
vis.add_geometry(pcd)

# Add coordinate axes at box_base origin (0,0,0) for spatial reference
axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5, origin=[0, 0, 0])
vis.add_geometry(axes)

for i in range(1, len(rgb_files)):
    new_pcd = build_pcd(rgb_files[i], depth_files[i])

    pcd.points = new_pcd.points
    pcd.colors = new_pcd.colors

    vis.update_geometry(pcd)
    if not vis.poll_events():
        break  # window was closed
    vis.update_renderer()

    time.sleep(frame_delay)
    print(f"\rFrame {i+1}/{len(rgb_files)}", end="")

print("\nDone.")
vis.destroy_window()

Found 4949 frame pairs
Frame 977/4949
Done.


: 